# Set up

In [ ]:
# import pandas as pd
import serial
import time
import numpy as np

from datetime import datetime
from pythonosc import dispatcher
from pythonosc import osc_server

# constants for arduino communication
BAUD = 9600
ARD_PORT = '/dev/cu.usbmodem48CA43547B4C2'

PLAYBACK_INTERVAL = 10

# constants for muse communication
MUSE_IP = "127.0.0.1"
MUSE_PORT = 5000

PLAYBACK_INTERVAL = 0.1 

# open and defining serial port for arduino
ser = serial.Serial(ARD_PORT, BAUD)
# give hardware time to reset
time.sleep(2)



# Variables

In [ ]:

# defining varaibles for each wave
wave_1 = 0
wave_2 = 0
wave_3 = 0
wave_4 = 0

alpha_max = 30
beta_max = 30
delta_max = 80
gamma_max = 60
theta_max = 60

tp9_data = []
af7_data = []
af8_data = []
tp10_data = []

collect_data = False

# list of values to see the percentage change to send to arduino
perVals = []

# Functions

In [ ]:
def mapping(minval, maxval, val):
    # this maps it between 0 and 1. depending on what i want the max to be, multiple this value by 255
    return (val - minval)/(maxval - minval)

In [ ]:
def send_arduino(data):
    # print("BLEHHH")
    global collect_data
    # cleaning? the data
    data = (data.split(",")[1:5])
    for t in range(0, len(data)):
        data[t] = int(float(data[t]))

    wave_1 = data[0]
    wave_2 = data[1]
    wave_3 = data[2]
    wave_4 = data[3]

    # adding each wave to its corresponding sensor
    tp9_data.append(wave_1) 
    af8_data.append(wave_2)
    af7_data.append(wave_3)
    tp10_data.append(wave_4)

    alpha_data = 0
    beta_data = 0
    delta_data = 0
    gamma_data = 0
    theta_data = 0

    buffers = [tp9_data, tp10_data, af7_data, af8_data]

    # if the buffers are full
    if collect_data == True:
        # loop through each buffer
        for buffer in buffers:
            # if the length is long enough for one (they all inc at the same time) do this
            if len(buffer) >= 256:
                # get a array starting from the 1st element, all the way to the end.
                signal = np.array(buffer[-256:])
                # for each signal value, remove the mean of the signal from it
                signal = signal - np.mean(signal)

                # fourier transform
                fft = np.fft.rfft(signal)
                freqs = np.fft.rfftfreq(len(signal), d=1/256) #frequency

                power = np.abs(fft)**2
                # get each wave using the band range. get the mean.
                # since this happens for each eeg channel, add the value to the wave value so they stack (makes averaging them easier)
                alpha_data += power[(freqs >= 8) & (freqs < 13)].mean()
                beta_data  += power[(freqs >= 13) & (freqs < 30)].mean()
                delta_data += power[(freqs >= 0.5) & (freqs < 4)].mean()
                gamma_data += power[(freqs >= 30) & (freqs <= 100)].mean()
                theta_data += power[(freqs >= 4) & (freqs < 8)].mean()

        # average of them after the totals are collected
        alpha_data /= 4
        beta_data  /= 4
        delta_data /= 4
        gamma_data /= 4
        theta_data /= 4

        total_power = alpha_data + beta_data + delta_data + gamma_data + theta_data
        
        # normalise
        alpha_norm = alpha_data / total_power
        beta_norm  = beta_data / total_power
        delta_norm = delta_data / total_power
        gamma_norm = gamma_data / total_power
        theta_norm = theta_data / total_power
        
        alpha_exag = np.sqrt((alpha_norm * 10000) * np.exp(5))
        beta_exag = np.sqrt((beta_norm * 10000) * np.exp(5))
        delta_exag = np.sqrt((delta_norm * 10000) * np.exp(5))
        gamma_exag = np.sqrt((gamma_norm * 10000) * np.exp(5))
        theta_exag = np.sqrt((theta_norm * 10000) * np.exp(5))


        perVals.append([alpha_exag, beta_exag, delta_exag, gamma_exag, theta_exag])

        if(len(perVals) >= 2):
            # calc percentage change
            sendingList = []
            for i in range(len(perVals[0])):
                    percentageChange = ((perVals[0][i] - perVals[1][i])/ perVals[1][i]) * 100
                    # do this to move decimal point so that im sending integers and also keep the extra bit of data <3
                    percentageChange = abs(percentageChange)
                    percentageChange = int((percentageChange * 100) * 10)
                    sendingList.append(percentageChange)
        
            print(f"A:{sendingList[0]}\nB:{sendingList[1]}\nD:{sendingList[2]}\nG:{sendingList[3]}\nT:{sendingList[4]}\n\n")
            ser.write(f"{sendingList[0]},{sendingList[1]},{sendingList[2]},{sendingList[3]},{sendingList[4]}\n".encode())

        time.sleep(0.5)

    if len(perVals) > 1:
        perVals.pop(0)
    if len(tp9_data) > 256:
        # this keeps updating the arrays but removes the first value so it stays fixed at 256 once its greater than it
        # since they all uodate at the same time, im just using one as the conditional
        collect_data = True
        tp9_data.pop(0)
        af8_data.pop(0)
        af7_data.pop(0)
        tp10_data.pop(0)

In [ ]:
def eeg_handler(address: str,*args):
    dateTimeObj = datetime.now()
    printStr = dateTimeObj.strftime("%Y-%m-%d %H:%M:%S.%f")
    addr = ""
    
    for arg in args:
        printStr += ","+str(arg)
        addr += ", "+str(address)

    send_arduino(printStr)

# Run

In [ ]:
if __name__ == "__main__":
    dispatcher = dispatcher.Dispatcher()
    dispatcher.map("/eeg", eeg_handler)

    server = osc_server.BlockingOSCUDPServer((MUSE_IP, MUSE_PORT), dispatcher)
    print("Listening on UDP port "+str(MUSE_PORT))

    server.serve_forever()



# close ports

In [ ]:
ser.close()